# Architecture 1 · Phase 2 · Legal NER

In [1]:
%pip -q install tokenizers
import os, json, math, random, time, gc, zipfile, shutil, copy, hashlib
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.utils.checkpoint import checkpoint
from tokenizers import Tokenizer
from IPython.display import display, FileLink
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
torch.set_num_threads(2)
BASE = Path('/kaggle/input/datasets/abdakt/p2-a1')
DATA = Path('/kaggle/input/datasets/abdakt/p2-a1/Phase_2_LNER')
OUT = Path('/kaggle/working/arch1_phase2_six')
OUT.mkdir(parents=True, exist_ok=True)
C = dict(seed=42, length=512, stride=384, batch=4, accum=4,
         epochs=12, patience=3, rank=16, alpha=16, bottleneck=64,
         dropout=0.1, weight_decay=0.01, temperature=2., kd_weight=0.5)
METHODS = ['full', 'partial', 'lora', 'lorma', 'adapter', 'kd']
LR = dict(full=2e-5, partial=4e-5, lora=3e-4, lorma=2e-4, adapter=3e-4, kd=4e-5)
device = torch.device('cuda:0')
def seed_all(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
def parallel(model):
    model = model.to(device)
    return nn.DataParallel(model, device_ids=list(range(torch.cuda.device_count())))
def cleanup():
    gc.collect(); torch.cuda.empty_cache()

Note: you may need to restart the kernel to use updated packages.


## Data and exact offsets

In [2]:
def data_file(name):
    return next(DATA.rglob(name))
def read_data(name):
    return json.loads(data_file(name).read_text())
labels = data_file('labels.txt').read_text().split()
tags = ['O'] + [p + '-' + label for label in labels for p in ['B', 'I']]
tag_id = {s:i for i,s in enumerate(tags)}
texts, gold, folds = {}, {}, []
for fold in [1, 2]:
    t, g = read_data(f'fold{fold}-text.json'), read_data(f'fold{fold}-gold.json')
    texts.update(t); gold.update(g); folds.append(sorted(t))
test_texts = read_data('fold3-text.json')
rng = random.Random(C['seed'])
train_ids, val_ids = [], []
for ids in folds:
    ids = ids.copy(); rng.shuffle(ids)
    val_ids += ids[:7]; train_ids += ids[7:]
tokenizer = Tokenizer.from_file(str(BASE / 'tokenizer.json'))
tokenizer.no_truncation(); tokenizer.no_padding()
pad_id = tokenizer.token_to_id('<pad>')
base_ckpt = torch.load(BASE / 'best.pt', map_location='cpu', weights_only=True)
base_cfg = base_ckpt['config']
base_state = base_ckpt['model']
del base_ckpt
encoded, alignment = {}, dict(entities=0, unaligned=0, overlapping=0)
for doc, text in {**texts, **test_texts}.items():
    e = tokenizer.encode(text, add_special_tokens=False)
    offsets = []
    for a,b in e.offsets:
        while a < b and text[a].isspace(): a += 1
        while b > a and text[b-1].isspace(): b -= 1
        offsets.append((a,b))
    y = np.zeros(len(e.ids), dtype=np.int64)
    starts = np.array([a for a,b in offsets]); ends = np.array([b for a,b in offsets])
    for a,b,label,*_ in sorted(gold.get(doc, [])):
        alignment['entities'] += 1
        ix = np.flatnonzero((ends > a) & (starts < b) & (ends > starts))
        if not len(ix):
            alignment['unaligned'] += 1; continue
        if offsets[ix[0]][0] != a or offsets[ix[-1]][1] != b:
            alignment['unaligned'] += 1
        if np.any(y[ix]):
            alignment['overlapping'] += 1; continue
        y[ix] = tag_id['I-' + label]; y[ix[0]] = tag_id['B-' + label]
    y[ends <= starts] = -100
    encoded[doc] = dict(ids=e.ids, offsets=offsets, y=y)
(OUT / 'alignment.json').write_text(json.dumps(alignment, indent=2))
(OUT / 'split.json').write_text(json.dumps(dict(train=train_ids, validation=val_ids), indent=2))
(OUT / 'settings.json').write_text(json.dumps(dict(training=C, lr=LR, labels=labels,
    base_config=base_cfg, base_path=str(BASE), tokenizer_sha256=hashlib.sha256(
    (BASE/'tokenizer.json').read_bytes()).hexdigest()), indent=2))
shutil.copy2(BASE / 'tokenizer.json', OUT / 'tokenizer.json')

class Windows(Dataset):
    def __init__(self, docs):
        self.rows = []
        for doc in docs:
            n = len(encoded[doc]['ids'])
            for start in range(0, n, C['stride']):
                self.rows.append((doc, start, min(start + C['length'], n)))
                if start + C['length'] >= n: break
        self.soft = None
    def __len__(self): return len(self.rows)
    def __getitem__(self, i): return i, self.rows[i]

def collate(rows):
    width = max(b-a for _,(_,a,b) in rows)
    ids = torch.full((len(rows),width), pad_id, dtype=torch.long)
    mask = torch.zeros_like(ids, dtype=torch.bool)
    y = torch.full_like(ids, -100)
    for j,(i,(doc,a,b)) in enumerate(rows):
        ids[j,:b-a] = torch.tensor(encoded[doc]['ids'][a:b])
        mask[j,:b-a] = True
        y[j,:b-a] = torch.tensor(encoded[doc]['y'][a:b])
    return ids, mask, y, [i for i,_ in rows]
def loader(ds, shuffle=False):
    return DataLoader(ds, batch_size=C['batch'], shuffle=shuffle, collate_fn=collate,
                      num_workers=0, pin_memory=True)
train_ds, val_ds, test_ds = Windows(train_ids), Windows(val_ids), Windows(list(test_texts))
print(f'{len(train_ids)} train / {len(val_ids)} validation / {len(test_texts)} test documents')

56 train / 14 validation / 35 test documents


## Architecture 1: LegalRoPE backbone

In [3]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.checkpoint import checkpoint


class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-5):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(dim))
        self.eps = eps

    def forward(self, x):
        y = x.float()
        y = y * torch.rsqrt(y.square().mean(-1, keepdim=True) + self.eps)
        return y.to(x.dtype) * self.weight


class Attention(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        d = cfg['dim']
        self.heads = cfg['heads']
        self.head_dim = d // self.heads
        self.qkv = nn.Linear(d, 3 * d, bias=False)
        self.out = nn.Linear(d, d, bias=False)
        inv = 1.0 / (10000 ** (torch.arange(0, self.head_dim, 2).float() / self.head_dim))
        self.register_buffer('inv_freq', inv, persistent=False)

    def rope(self, x, positions):
        phase = positions.float().unsqueeze(-1) * self.inv_freq.float()
        cos = phase.cos().unsqueeze(1).to(x.dtype)
        sin = phase.sin().unsqueeze(1).to(x.dtype)
        even, odd = x[..., ::2], x[..., 1::2]
        return torch.stack((even * cos - odd * sin, even * sin + odd * cos), -1).flatten(-2)

    def forward(self, x, positions, mask=None, past=None, use_cache=False):
        b, t, d = x.shape
        q, k, v = self.qkv(x).view(b, t, 3, self.heads, self.head_dim).unbind(2)
        q, k, v = (z.transpose(1, 2) for z in (q, k, v))
        q, k = self.rope(q, positions), self.rope(k, positions)
        old = 0
        if past is not None:
            old = past[0].size(2)
            k, v = torch.cat((past[0], k), 2), torch.cat((past[1], v), 2)
        allowed = None
        causal = False
        if mask is not None or old:
            key_idx = torch.arange(k.size(2), device=x.device)
            query_idx = old + torch.arange(t, device=x.device)
            allowed = torch.ones((t, k.size(2)), dtype=torch.bool, device=x.device)
            allowed = allowed[None, None, :, :]
            if mask is not None:
                allowed = allowed & mask[:, None, None, :].bool()
        y = F.scaled_dot_product_attention(q, k, v, attn_mask=allowed,
                                          dropout_p=0.0, is_causal=causal)
        y = self.out(y.transpose(1, 2).contiguous().view(b, t, d))
        return y, (k, v) if use_cache else None


class Block(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        d, f = cfg['dim'], cfg['ffn_dim']
        self.norm1, self.norm2 = RMSNorm(d), RMSNorm(d)
        self.attn = Attention(cfg)
        self.gate = nn.Linear(d, f, bias=False)
        self.up = nn.Linear(d, f, bias=False)
        self.down = nn.Linear(f, d, bias=False)

    def forward(self, x, positions, mask=None, past=None, use_cache=False):
        a, cache = self.attn(self.norm1(x), positions, mask, past, use_cache)
        x = x + a
        z = self.norm2(x)
        return x + self.down(F.silu(self.gate(z)) * self.up(z)), cache


class LegalLM(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = dict(cfg)
        self.embed = nn.Embedding(cfg['vocab_size'], cfg['dim'])
        self.blocks = nn.ModuleList([Block(cfg) for _ in range(cfg['layers'])])
        self.norm = RMSNorm(cfg['dim'])
        self.apply(self._init)
        for block in self.blocks:
            nn.init.normal_(block.attn.out.weight, std=0.02 / math.sqrt(2 * cfg['layers']))
            nn.init.normal_(block.down.weight, std=0.02 / math.sqrt(2 * cfg['layers']))

    @staticmethod
    def _init(module):
        if isinstance(module, (nn.Linear, nn.Embedding)):
            nn.init.normal_(module.weight, std=0.02)

    def forward(self, ids, targets=None, attention_mask=None, past=None,
                use_cache=False, last_only=False):
        old = 0 if past is None else past[0][0].size(2)
        if attention_mask is None:
            positions = torch.arange(old, old + ids.size(1), device=ids.device)[None, :]
        else:
            positions = (attention_mask.long().cumsum(-1) - 1).clamp_min(0)[:, -ids.size(1):]
        x, caches = self.embed(ids), []
        for i, block in enumerate(self.blocks):
            if self.training and self.cfg.get('gradient_checkpointing', False):
                x = checkpoint(lambda z, m=block: m(z, positions)[0], x, use_reentrant=False)
            else:
                x, cache = block(x, positions, attention_mask,
                                 None if past is None else past[i], use_cache)
                if use_cache:
                    caches.append(cache)
        x = self.norm(x)
        if targets is not None:
            h, y = x.flatten(0, 1), targets.flatten()
            total = h.new_zeros((), dtype=torch.float32)
            for a in range(0, h.size(0), 256):
                def part(z, labels):
                    return F.cross_entropy(F.linear(z, self.embed.weight).float(), labels, reduction='sum')
                if self.training:
                    loss = checkpoint(part, h[a:a+256], y[a:a+256], use_reentrant=False)
                else:
                    loss = part(h[a:a+256], y[a:a+256])
                total = total + loss
            return total / y.numel()
        logits = F.linear(x[:, -1:] if last_only else x, self.embed.weight)
        return logits, caches if use_cache else None

## Adaptation methods

In [4]:
class LowRank(nn.Module):
    def __init__(self, linear, multiplicative=False):
        super().__init__()
        self.base = linear
        self.multiplicative = multiplicative
        width = linear.out_features if multiplicative else linear.in_features
        self.A = nn.Linear(width, C['rank'], bias=False)
        self.B = nn.Linear(C['rank'], linear.out_features, bias=False)
        nn.init.kaiming_uniform_(self.A.weight, a=math.sqrt(5))
        nn.init.zeros_(self.B.weight)
        self.scale = C['alpha'] / C['rank']
    def forward(self, x):
        y = self.base(x)
        return y + self.B(self.A(y if self.multiplicative else x)) * self.scale

class Adapter(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.norm = RMSNorm(dim)
        self.down = nn.Linear(dim, C['bottleneck'])
        self.up = nn.Linear(C['bottleneck'], dim)
        nn.init.zeros_(self.up.weight); nn.init.zeros_(self.up.bias)
    def forward(self, x):
        return x + self.up(F.gelu(self.down(self.norm(x))))

class NER(nn.Module):
    def __init__(self, method):
        super().__init__()
        self.method = method
        cfg = dict(base_cfg)
        selected = list(range(1, cfg['layers'], 2)) if method == 'kd' else list(range(cfg['layers']))
        cfg['layers'] = len(selected)
        self.backbone = LegalLM(cfg)
        state = {k:v for k,v in base_state.items() if not k.startswith('blocks.')}
        for new, old in enumerate(selected):
            prefix = f'blocks.{old}.'
            state.update({f'blocks.{new}.' + k[len(prefix):]:v for k,v in base_state.items()
                          if k.startswith(prefix)})
        self.backbone.load_state_dict(state, strict=True)
        self.backbone.requires_grad_(method in ['full', 'kd'])
        if method == 'partial':
            for block in self.backbone.blocks[-6:]: block.requires_grad_(True)
            self.backbone.norm.requires_grad_(True)
        if method in ['lora', 'lorma']:
            for block in self.backbone.blocks:
                for parent, name in [(block.attn,'qkv'), (block.attn,'out'),
                                     (block,'gate'), (block,'up'), (block,'down')]:
                    setattr(parent, name, LowRank(getattr(parent,name), method == 'lorma'))
        self.adapters = nn.ModuleList([Adapter(cfg['dim']) for _ in selected]) if method == 'adapter' else None
        self.drop = nn.Dropout(C['dropout'])
        self.head = nn.Linear(cfg['dim'], len(tags))
    def forward(self, ids, mask):
        positions = torch.arange(ids.shape[1], device=ids.device)[None,:]
        x = self.backbone.embed(ids)
        for i, block in enumerate(self.backbone.blocks):
            def layer(z, block=block, i=i):
                h = block(z, positions, mask)[0]
                return self.adapters[i](h) if self.adapters is not None else h
            x = checkpoint(layer, x, use_reentrant=False) if self.training else layer(x)
        return self.head(self.drop(self.backbone.norm(x)))

def save_model(model, path, info):
    trainable = {n for n,p in model.named_parameters() if p.requires_grad}
    state = {n:p.detach().cpu().clone() for n,p in model.state_dict().items() if n in trainable}
    temp = path.with_suffix('.tmp')
    torch.save(dict(model=state, **info), temp); os.replace(temp,path)
def load_model(path):
    saved = torch.load(path, map_location='cpu', weights_only=True)
    model = NER(saved['method'])
    model.load_state_dict(saved['model'], strict=False)
    return model, saved

## Span decoding and validation

In [5]:
def decode(doc, prediction):
    offsets = encoded[doc]['offsets']
    spans, active = [], None
    for i,t in enumerate(prediction):
        a,b = offsets[i]
        if a == b: continue
        tag = tags[int(t)]
        if tag == 'O':
            if active is not None: spans.append(active); active = None
            continue
        prefix,label = tag.split('-',1)
        if prefix == 'B' or active is None or active[2] != label:
            if active is not None: spans.append(active)
            active = [a,b,label]
        else:
            active[1] = b
    if active is not None: spans.append(active)
    return [list(x) for x in sorted(set(tuple(x) for x in spans))]

@torch.inference_mode()
def predict(net, ds):
    net.eval()
    docs = list(dict.fromkeys(d for d,a,b in ds.rows))
    pred = {d:np.zeros(len(encoded[d]['ids']),dtype=np.int64) for d in docs}
    priority = {d:np.full(len(encoded[d]['ids']),-1) for d in docs}
    for ids,mask,_,indices in loader(ds):
        with torch.autocast('cuda', dtype=torch.float16):
            logits = net(ids.to(device),mask.to(device))
        guesses = logits.argmax(-1).cpu().numpy()
        for j,index in enumerate(indices):
            d,a,b = ds.rows[index]; n=b-a
            score = np.minimum(np.arange(n),np.arange(n)[::-1])
            take = score > priority[d][a:b]
            pred[d][a:b][take] = guesses[j,:n][take]
            priority[d][a:b][take] = score[take]
    return {d:decode(d,pred[d]) for d in docs}

def metrics(pred):
    per_label = {}
    for label in labels:
        p = {(d,int(a),int(b)) for d,spans in pred.items() for a,b,l,*_ in spans if l==label}
        g = {(d,int(a),int(b)) for d in pred for a,b,l,*_ in gold[d] if l==label}
        tp = len(p & g); precision = tp/max(1,len(p)); recall = tp/max(1,len(g))
        per_label[label] = dict(precision=precision, recall=recall,
                               f1=2*tp/max(1,len(p)+len(g)), gold=len(g), predicted=len(p))
    return dict(macro_f1=float(np.mean([v['f1'] for v in per_label.values()])),
                macro_precision=float(np.mean([v['precision'] for v in per_label.values()])),
                macro_recall=float(np.mean([v['recall'] for v in per_label.values()])), per_label=per_label)

def submission(net, method, folder=OUT):
    pred = predict(net,test_ds)
    path = folder / f'prediction_{method}.json'
    path.write_text(json.dumps(pred,ensure_ascii=False))
    with zipfile.ZipFile(folder/f'submission_{method}.zip','w',zipfile.ZIP_DEFLATED) as z:
        z.write(path,'prediction.json')

@torch.inference_mode()
def cache_teacher(path, ds):
    model,_ = load_model(path); net = parallel(model); net.eval()
    cache = [None] * len(ds)
    for ids,mask,_,indices in loader(ds):
        with torch.autocast('cuda',dtype=torch.float16):
            logits = net(ids.to(device),mask.to(device)).half().cpu()
        for j,index in enumerate(indices):
            _,a,b = ds.rows[index]; cache[index] = logits[j,:b-a].clone()
    del net,model; cleanup()
    return cache

## Training

In [6]:
def fit(method, ds, folder, soft=None, fixed_epochs=None):
    seed_all(C['seed'])
    folder.mkdir(parents=True, exist_ok=True)
    model = NER(method); net = parallel(model)
    backbone, head = [], []
    for name,p in model.named_parameters():
        if p.requires_grad:
            (head if name.startswith('head.') else backbone).append(p)
    optimizer = torch.optim.AdamW([dict(params=backbone,lr=LR[method]),
                                   dict(params=head,lr=3e-4)], weight_decay=C['weight_decay'])
    scaler = torch.amp.GradScaler('cuda')
    counts = np.zeros(len(tags))
    for d in dict.fromkeys(d for d,a,b in ds.rows):
        y = encoded[d]['y']; counts += np.bincount(y[y>=0],minlength=len(tags))
    weights = torch.tensor(np.clip(np.sqrt(max(counts[0],1)/np.maximum(counts,1)),1,8),
                           dtype=torch.float32, device=device)
    train_loader = loader(ds,True)
    epochs = fixed_epochs or C['epochs']
    total_steps = epochs * math.ceil(len(train_loader)/C['accum'])
    warmup = max(1,int(total_steps*.1)); step = 0
    best, stale, history = -1., 0, []
    start = time.monotonic()
    for epoch in range(1,epochs+1):
        net.train(); optimizer.zero_grad(set_to_none=True); losses=[]
        for batch,(ids,mask,y,indices) in enumerate(train_loader):
            ids,mask,y = ids.to(device),mask.to(device),y.to(device)
            group_start = (batch//C['accum'])*C['accum']
            group_size = min(C['accum'],len(train_loader)-group_start)
            with torch.autocast('cuda',dtype=torch.float16):
                logits = net(ids,mask)
                loss = F.cross_entropy(logits.float().reshape(-1,len(tags)),y.flatten(),weight=weights)
                if soft is not None:
                    target = torch.zeros_like(logits,dtype=torch.float32)
                    for j,index in enumerate(indices):
                        target[j,:len(soft[index])] = soft[index].to(device)
                    t = C['temperature']; valid = (y != -100)
                    kl = F.kl_div(F.log_softmax(logits.float()/t,-1),
                                  F.softmax(target/t,-1),reduction='none').sum(-1)
                    kd = (kl*valid).sum()/valid.sum().clamp_min(1)*t*t
                    loss = (1-C['kd_weight'])*loss + C['kd_weight']*kd
            scaler.scale(loss/group_size).backward(); losses.append(loss.item())
            if (batch+1)%C['accum']==0 or batch+1==len(train_loader):
                factor = min((step+1)/warmup,1.) if step<warmup else .1+.9*.5*(1+math.cos(math.pi*(step-warmup)/max(1,total_steps-warmup)))
                optimizer.param_groups[0]['lr'] = LR[method]*factor
                optimizer.param_groups[1]['lr'] = 3e-4*factor
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(),1.)
                scaler.step(optimizer); scaler.update(); optimizer.zero_grad(set_to_none=True); step+=1
        row = dict(epoch=epoch,loss=float(np.mean(losses)))
        if fixed_epochs is None:
            result = metrics(predict(net,val_ds)); row.update(result)
            improved = result['macro_f1'] > best
            if improved: best=result['macro_f1']; stale=0
            else: stale+=1
        else:
            improved = epoch==epochs
        history.append(row)
        if improved:
            save_model(model,folder/'best.pt',dict(method=method,epoch=epoch,
                       validation_f1=best,config=C,labels=labels))
        (folder/'history.json').write_text(json.dumps(history,indent=2))
        suffix = f" F1={row['macro_f1']:.4f}" if fixed_epochs is None else ''
        print(f"{method}: epoch {epoch} loss={row['loss']:.4f}{suffix}",flush=True)
        if fixed_epochs is None and stale>=C['patience']: break
    stats = dict(method=method,parameters=sum(p.numel() for p in model.parameters()),
                 trainable=sum(p.numel() for p in model.parameters() if p.requires_grad),
                 minutes=(time.monotonic()-start)/60)
    del net,model,optimizer,scaler; cleanup()
    model,saved = load_model(folder/'best.pt'); net = parallel(model)
    submission(net,method,folder.parent)
    if fixed_epochs is None:
        pred = predict(net,val_ds)
        (folder/'validation_predictions.json').write_text(json.dumps(pred))
        stats.update(metrics(pred)); stats['best_epoch']=saved['epoch']
    del net,model; cleanup()
    (folder/'done.json').write_text(json.dumps(stats,indent=2))
    return stats

results=[]
for method in METHODS:
    folder=OUT/method
    if (folder/'done.json').exists():
        results.append(json.loads((folder/'done.json').read_text())); continue
    soft=None
    if method=='kd':
        teacher=max(results,key=lambda r:r['macro_f1'])['method']
        print('Distillation teacher:',teacher)
        (OUT/'teacher.json').write_text(json.dumps(dict(method=teacher)))
        soft=cache_teacher(OUT/teacher/'best.pt',train_ds)
    results.append(fit(method,train_ds,folder,soft))
    del soft; cleanup()
(OUT/'comparison.json').write_text(json.dumps(results,indent=2))
for r in results:
    print(f"{r['method']:8} F1={r['macro_f1']:.4f} parameters={r['parameters']:,} trainable={r['trainable']:,}")
for method in METHODS: display(FileLink(str(OUT/f'submission_{method}.zip')))

full: epoch 1 loss=1.5225 F1=0.2145
full: epoch 2 loss=0.1915 F1=0.2868
full: epoch 3 loss=0.0673 F1=0.3083
full: epoch 4 loss=0.0316 F1=0.3416
full: epoch 5 loss=0.0148 F1=0.3390
full: epoch 6 loss=0.0069 F1=0.3474
full: epoch 7 loss=0.0038 F1=0.3434
full: epoch 8 loss=0.0019 F1=0.3335
full: epoch 9 loss=0.0010 F1=0.3429
partial: epoch 1 loss=1.4980 F1=0.2079
partial: epoch 2 loss=0.2187 F1=0.3095
partial: epoch 3 loss=0.0924 F1=0.3265
partial: epoch 4 loss=0.0489 F1=0.3278
partial: epoch 5 loss=0.0309 F1=0.3725
partial: epoch 6 loss=0.0208 F1=0.3527
partial: epoch 7 loss=0.0111 F1=0.3686
partial: epoch 8 loss=0.0066 F1=0.3651
lora: epoch 1 loss=1.8708 F1=0.1325
lora: epoch 2 loss=0.3067 F1=0.2901
lora: epoch 3 loss=0.1296 F1=0.3276
lora: epoch 4 loss=0.0760 F1=0.3558
lora: epoch 5 loss=0.0456 F1=0.3641
lora: epoch 6 loss=0.0271 F1=0.3694
lora: epoch 7 loss=0.0183 F1=0.3666
lora: epoch 8 loss=0.0131 F1=0.3689
lora: epoch 9 loss=0.0080 F1=0.3446
lorma: epoch 1 loss=2.0874 F1=0.1254
lor

/kaggle/working/arch1_phase2_six/submission_full.zip

/kaggle/working/arch1_phase2_six/submission_partial.zip

/kaggle/working/arch1_phase2_six/submission_lora.zip

/kaggle/working/arch1_phase2_six/submission_lorma.zip

/kaggle/working/arch1_phase2_six/submission_adapter.zip

/kaggle/working/arch1_phase2_six/submission_kd.zip

## Full-data refit

In [7]:
refit = OUT/'refit'; refit.mkdir(exist_ok=True)
all_ds = Windows(sorted(texts))
teacher = json.loads((OUT/'teacher.json').read_text())['method']
for method in METHODS:
    if (refit/method/'done.json').exists(): continue
    epochs = next(r['best_epoch'] for r in results if r['method']==method)
    soft = cache_teacher(refit/teacher/'best.pt',all_ds) if method=='kd' else None
    fit(method,all_ds,refit/method,soft,fixed_epochs=epochs)
    del soft; cleanup()
for method in METHODS: display(FileLink(str(refit/f'submission_{method}.zip')))

full: epoch 1 loss=1.0941
full: epoch 2 loss=0.1287
full: epoch 3 loss=0.0513
full: epoch 4 loss=0.0242
full: epoch 5 loss=0.0114
full: epoch 6 loss=0.0056
partial: epoch 1 loss=1.0169
partial: epoch 2 loss=0.1436
partial: epoch 3 loss=0.0685
partial: epoch 4 loss=0.0429
partial: epoch 5 loss=0.0300
lora: epoch 1 loss=1.3899
lora: epoch 2 loss=0.2142
lora: epoch 3 loss=0.1122
lora: epoch 4 loss=0.0688
lora: epoch 5 loss=0.0441
lora: epoch 6 loss=0.0317
lorma: epoch 1 loss=1.6225
lorma: epoch 2 loss=0.2321
lorma: epoch 3 loss=0.1239
lorma: epoch 4 loss=0.0778
lorma: epoch 5 loss=0.0536
lorma: epoch 6 loss=0.0357
lorma: epoch 7 loss=0.0265
adapter: epoch 1 loss=1.2110
adapter: epoch 2 loss=0.2370
adapter: epoch 3 loss=0.1357
adapter: epoch 4 loss=0.0887
adapter: epoch 5 loss=0.0665
kd: epoch 1 loss=1.9833
kd: epoch 2 loss=0.4619
kd: epoch 3 loss=0.2030
kd: epoch 4 loss=0.1242
kd: epoch 5 loss=0.0830
kd: epoch 6 loss=0.0578
kd: epoch 7 loss=0.0437
kd: epoch 8 loss=0.0356
kd: epoch 9 loss=

/kaggle/working/arch1_phase2_six/refit/submission_full.zip

/kaggle/working/arch1_phase2_six/refit/submission_partial.zip

/kaggle/working/arch1_phase2_six/refit/submission_lora.zip

/kaggle/working/arch1_phase2_six/refit/submission_lorma.zip

/kaggle/working/arch1_phase2_six/refit/submission_adapter.zip

/kaggle/working/arch1_phase2_six/refit/submission_kd.zip

In [8]:
@torch.inference_mode()
def ensemble_predict(ds):
    docs = list(dict.fromkeys(d for d, _, _ in ds.rows))
    totals = {
        d: np.zeros((len(encoded[d]['ids']), len(tags)), dtype=np.float32)
        for d in docs
    }

    for method in METHODS:
        model, _ = load_model(OUT / method / 'best.pt')
        net = parallel(model)
        net.eval()

        scores = {d: np.zeros_like(totals[d]) for d in docs}
        priority = {
            d: np.full(len(encoded[d]['ids']), -1)
            for d in docs
        }

        for ids, mask, _, indices in loader(ds):
            with torch.autocast('cuda', dtype=torch.float16):
                logits = net(ids.to(device), mask.to(device))
            probs = logits.float().softmax(-1).cpu().numpy()

            for j, index in enumerate(indices):
                d, a, b = ds.rows[index]
                n = b - a
                context = np.minimum(np.arange(n), np.arange(n)[::-1])
                take = context > priority[d][a:b]
                scores[d][a:b][take] = probs[j, :n][take]
                priority[d][a:b][take] = context[take]

        for d in docs:
            totals[d] += scores[d] / len(METHODS)

        del net, model, scores
        cleanup()
        print(f'Ensemble: {method} complete')

    return {
        d: decode(d, totals[d].argmax(-1))
        for d in docs
    }

In [9]:
ensemble_val = ensemble_predict(val_ds)
ensemble_metrics = metrics(ensemble_val)

(OUT / 'ensemble_metrics.json').write_text(
    json.dumps(ensemble_metrics, indent=2)
)
(OUT / 'ensemble_validation_predictions.json').write_text(
    json.dumps(ensemble_val, ensure_ascii=False)
)
print(f"Ensemble validation F1: {ensemble_metrics['macro_f1']:.4f}")

ensemble_test = ensemble_predict(test_ds)
prediction_path = OUT / 'prediction_ensemble.json'
prediction_path.write_text(json.dumps(ensemble_test, ensure_ascii=False))

with zipfile.ZipFile(
    OUT / 'submission_ensemble.zip', 'w', zipfile.ZIP_DEFLATED
) as z:
    z.write(prediction_path, arcname='prediction.json')

display(FileLink(str(OUT / 'submission_ensemble.zip')))

Ensemble: full complete
Ensemble: partial complete
Ensemble: lora complete
Ensemble: lorma complete
Ensemble: adapter complete
Ensemble: kd complete
Ensemble validation F1: 0.4167
Ensemble: full complete
Ensemble: partial complete
Ensemble: lora complete
Ensemble: lorma complete
Ensemble: adapter complete
Ensemble: kd complete


/kaggle/working/arch1_phase2_six/submission_ensemble.zip

In [10]:
for path in OUT.rglob('*.zip'):
    with zipfile.ZipFile(path) as z:
        if z.namelist() != ['prediction.json']:
            continue
        pred = json.loads(z.read('prediction.json'))

    missing = set(test_texts) - set(pred)
    if not missing:
        print(path.name, 'OK')
        continue

    for doc_id in missing:
        if test_texts[doc_id] == '':
            pred[doc_id] = []

    remaining = set(test_texts) - set(pred)
    if remaining:
        print(path.name, 'Needs predictions for:', sorted(remaining))
        continue

    with zipfile.ZipFile(path, 'w', zipfile.ZIP_DEFLATED) as z:
        z.writestr('prediction.json', json.dumps(pred, ensure_ascii=False))

    print(path.name, 'fixed:', sorted(missing))

submission_lora.zip fixed: ['1305957']
submission_kd.zip fixed: ['1305957']
submission_adapter.zip fixed: ['1305957']
submission_lorma.zip fixed: ['1305957']
submission_ensemble.zip fixed: ['1305957']
submission_partial.zip fixed: ['1305957']
submission_full.zip fixed: ['1305957']
submission_lora.zip fixed: ['1305957']
submission_kd.zip fixed: ['1305957']
submission_adapter.zip fixed: ['1305957']
submission_lorma.zip fixed: ['1305957']
submission_partial.zip fixed: ['1305957']
submission_full.zip fixed: ['1305957']


## Outputs